# Data Cleaning

In [ ]:
!pip install -q evaluate rouge_score bert_score transformers datasets peft accelerate bitsandbytes trl jsonlines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.9 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import json
import zipfile
import math
import re
import random
from datasets import load_dataset, Dataset, DatasetDict
import evaluate
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from transformers import pipeline
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# from peft.utils import enable_input_require_grads
import torch
from google.colab import files
from tqdm import tqdm

# Evaluation metrics
from bert_score import score as bert_score

# optional: to quiet HF warnings
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
print(torch.cuda.is_available())

True


In [ ]:
uploaded = files.upload()

Saving tips_dataset.csv to tips_dataset.csv


In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
def clean_dataset(file_path, save_cleaned=True):
    print(f"\nProcessing: {file_path}")

    # Read dataset
    df = pd.read_csv(file_path, encoding='utf-8')

    relevant_cols = [
        "id", "author", "body",
        "openness", "conscientiousness", "extraversion",
        "agreeableness", "neuroticism", "dominant"
    ]
    existing_cols = [c for c in relevant_cols if c in df.columns]
    df = df[existing_cols].copy()
    print(f"Columns retained: {existing_cols}")

    # Relevant keywords
    keywords = [
        "eat", "eating", "ate", "food", "meal", "snack", "diet", "hungry", "hunger",
        "breakfast", "lunch", "dinner", "cook", "cooking", "recipe", "bake", "grill",
        "restaurant", "taste", "nutrition", "calorie", "healthy", "unhealthy",
        "fasting", "weight", "body", "fitness", "craving", "appetite", "mindful"
    ]
    pattern = re.compile("|".join(keywords), re.IGNORECASE)

    # Filter rows related to eating/food
    if "body" in df.columns:
        df = df[df["body"].astype(str).apply(lambda x: bool(pattern.search(x)))]
    else:
        print("No 'body' column found. Skipping keyword filtering.")

    # Cleaning function
    def clean_text(text):
        text = str(text)
        text = re.sub(r"http\S+|www\S+", "", text)  # Remove URLs
        text = re.sub(r"[^A-Za-z\s]", " ", text)    # Remove special chars & numbers
        text = re.sub(r"\s+", " ", text).strip()    # Remove extra spaces
        return text.lower()

    if "body" in df.columns:
        df["body"] = df["body"].apply(clean_text)

    # Dropping duplicates and blanks
    df = df.drop_duplicates(subset=["body"]) if "body" in df.columns else df
    df = df[df["body"].str.strip() != ""] if "body" in df.columns else df

    print(f"After cleaning: {df.shape}")

    # Saving the cleaned file
    if save_cleaned:
        out_path = file_path.replace(".csv", "_cleaned.csv")
        df.to_csv(out_path, index=False)
        print(f"Saved cleaned file: {out_path}")

    return df

In [ ]:
def clean_all_datasets(folder_path):
    csv_files = glob.glob(f"{folder_path}/*.csv")
    all_cleaned = []

    for file in csv_files:
        cleaned_df = clean_dataset(file)
        all_cleaned.append(cleaned_df)

    print(f"\nAll files processed. Total datasets cleaned: {len(all_cleaned)}")
    return all_cleaned

In [ ]:
def merge_cleaned_datasets(folder_path, output_name="combined_cleaned_dataset.csv"):
    cleaned_files = glob.glob(f"{folder_path}/*_cleaned.csv")
    combined_df = pd.concat([pd.read_csv(f) for f in cleaned_files], ignore_index=True)
    combined_df.to_csv(output_name, index=False)
    print(f"\nCombined dataset saved as: {output_name}")
    return combined_df

In [ ]:
uploaded = files.upload()

# Unzip to /content/datasets
with zipfile.ZipFile("Mindful_Eating_Datasets.zip", "r") as zip_ref:
    zip_ref.extractall("/content/datasets")

print("Folder extracted at /content/datasets")

Saving Mindful_Eating_Datasets.zip to Mindful_Eating_Datasets.zip
✅ Folder extracted at /content/datasets


In [ ]:
folder_path = "/content/datasets/Mindful_Eating_Datasets"
all_cleaned = clean_all_datasets(folder_path)
combined_df = merge_cleaned_datasets(folder_path)
files.download("combined_cleaned_dataset.csv")
print("Cleaned and merged all datasets.")


🔹 Processing: /content/datasets/Mindful_Eating_Datasets/data_scored3.csv
✅ Columns retained: ['id', 'author', 'body', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'dominant']
✅ After cleaning: (1931, 9)
💾 Saved cleaned file: /content/datasets/Mindful_Eating_Datasets/data_scored3_cleaned.csv

🔹 Processing: /content/datasets/Mindful_Eating_Datasets/data_scored4.csv
✅ Columns retained: ['id', 'author', 'body', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'dominant']
✅ After cleaning: (1617, 9)
💾 Saved cleaned file: /content/datasets/Mindful_Eating_Datasets/data_scored4_cleaned.csv

🔹 Processing: /content/datasets/Mindful_Eating_Datasets/data_scored8.csv
✅ Columns retained: ['id', 'author', 'body', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'dominant']
✅ After cleaning: (1568, 9)
💾 Saved cleaned file: /content/datasets/Mindful_Eating_Datasets/data_scored8_cleaned.csv

🔹 Processing

In [ ]:
combined_df.shape

(8042, 9)

In [ ]:
print(combined_df.isnull().sum())

id                   0
author               0
body                 0
openness             0
conscientiousness    0
extraversion         0
agreeableness        0
neuroticism          0
dominant             0
dtype: int64


#Phase 1: Pre-processing & Prompt Formatting

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("combined_cleaned_dataset.csv")
print("Dataset loaded:", df.shape)
df.head()

Saving combined_cleaned_dataset.csv to combined_cleaned_dataset.csv
Dataset loaded: (8042, 9)


,id,author,body,openness,conscientiousness,extraversion,agreeableness,neuroticism,dominant
0,fconpld,luciliddream,i m craving some right now with hot mustard and vinegar black pepper,0.290090,0.544630,0.555523,0.313025,0.585026,neuroticism
1,fconslv,ajoltman,oh used wheat flour instead of bleached all purpose,0.617622,0.609814,0.294844,0.648310,0.302407,agreeableness
2,fconw4g,lexoman323,i eat the pickle and tomatoes separate then go ham on the rest of the hotdog,0.553910,0.338326,0.312299,0.357127,0.546118,openness
3,fconxjv,Bifrons,you missed a golden opportunity to eat beans by the bean,0.244822,0.244062,0.270274,0.336682,0.612364,neuroticism
4,fcoo3ra,Afeazo,as someone who likes ketchup but never on a hotdog i think it s just a widely accepted flavor profile that doesn t work similar to how i like grilled onions and i like grilled cheese but i m not going to stick grilled onions in my grilled cheese plus a chicago style dog is a little spicy with the,0.614804,0.641645,0.295540,0.364308,0.659647,neuroticism


We’ll convert each record into the following template:

**Instruction:**
Based on this user's text and personality traits, generate a short, personalized mindful-eating tip.

**Input:**
Text: "<body>"
Traits: openness=..., conscientiousness=..., extraversion=..., agreeableness=..., neuroticism=...

**Response:**
expected mindful eating advice text

In [ ]:
def create_prompt(row):
    instruction = (
        "Based on this user's text and personality traits, "
        "generate a short, personalized mindful-eating tip."
    )

    input_text = (
        f"Text: \"{row['body']}\"\n"
        f"Traits: openness={row['openness']}, conscientiousness={row['conscientiousness']}, "
        f"extraversion={row['extraversion']}, agreeableness={row['agreeableness']}, "
        f"neuroticism={row['neuroticism']}"
    )

    # Placeholder target (will be learned during SFT)
    response = "Generate a mindful eating tip based on above."

    return {
        "instruction": instruction,
        "input": input_text,
        "output": response
    }

In [ ]:
# Apply function
data = df.apply(create_prompt, axis=1).tolist()
print("Prompt formatting complete! Total records:", len(data))
print(json.dumps(data[0], indent=2))

Prompt formatting complete! Total records: 8042
{
  "instruction": "Based on this user's text and personality traits, generate a short, personalized mindful-eating tip.",
  "input": "Text: \"i m craving some right now with hot mustard and vinegar black pepper\"\nTraits: openness=0.2900896579104082, conscientiousness=0.544630450605439, extraversion=0.5555233904585456, agreeableness=0.3130252100840336, neuroticism=0.5850261364388275",
  "output": "Generate a mindful eating tip based on above."
}


In [ ]:
train_data, val_data = train_test_split(data, test_size=0.1, random_state=42)
print(f"Train size: {len(train_data)}, Validation size: {len(val_data)}")

Train size: 7237, Validation size: 805


In [ ]:
with jsonlines.open("train.jsonl", mode="w") as writer:
    writer.write_all(train_data)

with jsonlines.open("val.jsonl", mode="w") as writer:
    writer.write_all(val_data)

print("train.jsonl and val.jsonl saved successfully!")

train.jsonl and val.jsonl saved successfully!


In [ ]:
!head -n 20 train.jsonl

{"instruction": "Based on this user's text and personality traits, generate a short, personalized mindful-eating tip.", "input": "Text: \"i ate some good ribs from johore bahru\"\nTraits: openness=0.3860444245007853, conscientiousness=0.607134844065515, extraversion=0.3567795976366764, agreeableness=0.592543564430484, neuroticism=0.5894622690898214", "output": "Generate a mindful eating tip based on above."}
{"instruction": "Based on this user's text and personality traits, generate a short, personalized mindful-eating tip.", "input": "Text: \"the scale is my enemy the scale determines what i m going to eat and it s sad some days i can t even drink a lot of water when i m obsessively checking my weight throughout the day because i get so put down when i see the scale go up i know it s just water weight but some fucked up part of my hea\"\nTraits: openness=0.6461314417249965, conscientiousness=0.6535913938084276, extraversion=0.2929393526565509, agreeableness=0.3379903227321839, neuroti

#Phase 2 – Training and Fine-tuning LLM models

##TinyLlama on Correct dataset (tips_dataset.csv)

In [ ]:
df = pd.read_csv("tips_dataset.csv")
df.sample(5)

,Behavior_Big5,Message
312,"{'Eating Behavior': 'Fasting or Restrictive Eating Patterns', 'Dominant Big5': 'Agreeableness'}",Encourage a balanced approach to eating in your community—healthy habits grow stronger together.
758,"{'Eating Behavior': 'Mindful vs. Distracted Eating', 'Dominant Big5': 'Agreeableness'}",Mindful eating helps you stay in tune with your body’s needs. Trust your instincts and listen to your hunger and fullness cues.
1408,"{'Eating Behavior': 'Speed of Eating', 'Dominant Big5': 'Conscientiousness'}",Consider using a mindful eating app to track and improve your eating speed over time.
1126,"{'Eating Behavior': 'Snacking Habits', 'Dominant Big5': 'Extraversion'}","If you snack while socializing, choose foods that are both satisfying and nutritious."
569,"{'Eating Behavior': 'Hydration Practices', 'Dominant Big5': 'Neuroticism'}",Hydration is one of the simplest ways to support your well-being—start with a single glass today.


In [ ]:
# Extract EatingBehavior and DominantTrait from Behavior_Big5 if needed
def extract_fields(x):
    # If Behavior_Big5 is a stringified dict, try to eval safely
    try:
        if isinstance(x, str) and ("{" in x or ":" in x):
            d = eval(x)
            eb = d.get("Eating Behavior") or d.get("EatingBehavior") or d.get("Eating_behaviour")
            dt = d.get("Dominant Big5") or d.get("DominantBig5") or d.get("Dominant_Big5")
            return eb, dt
    except Exception:
        pass
    return None, None

# If Behavior_Big5 exists, parse; otherwise assume columns available already
if "Behavior_Big5" in df.columns:
    df["EatingBehavior"], df["DominantTrait"] = zip(*df["Behavior_Big5"].map(extract_fields))
else:
    # ensure the expected columns exist
    if "EatingBehavior" not in df.columns or "DominantTrait" not in df.columns:
        raise ValueError("tips_dataset.csv missing EatingBehavior / DominantTrait fields")

# Filter and keep required columns
df = df[["EatingBehavior", "DominantTrait", "Message"]].dropna().reset_index(drop=True)
print("Rows remaining:", len(df))

Rows remaining: 1500


In [ ]:
# Build instruction-format examples
def make_example(row):
    ins = "Generate a short, personalized mindful-eating tip based on the user's dominant trait and selected eating behavior."
    inp = f"Dominant Trait: {row['DominantTrait']}\nEating Behavior: {row['EatingBehavior']}"
    out = row["Message"].strip()
    return {"instruction": ins, "input": inp, "output": out}

data = [make_example(r) for _, r in df.iterrows()]

# Train/val split
train, val = train_test_split(data, test_size=0.1, random_state=42, shuffle=True)
print(f"Train: {len(train)}, Val: {len(val)}")

# Convert to HuggingFace Dataset
train_ds = Dataset.from_list(train)
val_ds = Dataset.from_list(val)
dataset = DatasetDict({"train": train_ds, "validation": val_ds})

Train: 1350, Val: 150


In [ ]:
# Tokenization
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_prompt(example):
    return (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n{example['output']}"
    )

def tokenize_fn(example):
    text = format_prompt(example)
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized = dataset.map(lambda ex: tokenize_fn(ex), batched=False)
tokenized = tokenized.remove_columns([c for c in tokenized["train"].column_names if c not in ["input_ids","attention_mask","labels"]]) if "input_ids" in tokenized["train"].column_names else tokenized
print(tokenized)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Map:   0%|          | 0/1350 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1350
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 150
    })
})


In [ ]:
# Load model in fp16; device_map auto
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Conservative LoRA config
lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # wider coverage
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

try:
    model.gradient_checkpointing_disable()
except Exception:
    model.config.use_cache = False

# Ensuring model in train mode
model.train()

# Explicitly set requires_grad: only LoRA params True
num_requiring = 0
num_total = 0
for name, p in model.named_parameters():
    num_total += p.numel()
    if ("lora" in name.lower()) or ("LORA" in name) or ("adapter" in name.lower()):
        p.requires_grad = True
        num_requiring += p.numel()
    else:
        p.requires_grad = False

print(f"Trainable params: {num_requiring:,}  Total params: {num_total:,}  Percent: {100*num_requiring/num_total:.6f}%")

# List a sample of trainable parameter names to verify
trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
print("Sample trainable names:", trainable_names[:40])

# Final safety check
if num_requiring == 0:
    raise RuntimeError("No LoRA parameters found as trainable. Check target_modules names or LoRA attachment.")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Trainable params: 1,126,400  Total params: 1,101,174,784  Percent: 0.102291%
Sample trainable names: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.lora_B.default.weight', 'base_mode

In [ ]:
training_args = TrainingArguments(
    output_dir="./tinyllama-lora-tips",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    # max_steps=1,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="no",
    warmup_ratio=0.03,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

In [ ]:
trainer.train()

{'loss': 2.0716, 'grad_norm': 2.4737679958343506, 'learning_rate': 0.00019877800407331977, 'epoch': 0.11851851851851852}
{'loss': 0.7054, 'grad_norm': 1.140821099281311, 'learning_rate': 0.00019063136456211815, 'epoch': 0.23703703703703705}
{'loss': 0.5208, 'grad_norm': 1.2494410276412964, 'learning_rate': 0.0001824847250509165, 'epoch': 0.35555555555555557}
{'loss': 0.5098, 'grad_norm': 1.079506278038025, 'learning_rate': 0.00017433808553971486, 'epoch': 0.4740740740740741}
{'loss': 0.4974, 'grad_norm': 1.0434198379516602, 'learning_rate': 0.00016619144602851324, 'epoch': 0.5925925925925926}
{'loss': 0.4782, 'grad_norm': 1.2393354177474976, 'learning_rate': 0.00015804480651731163, 'epoch': 0.7111111111111111}
{'loss': 0.487, 'grad_norm': 1.1837722063064575, 'learning_rate': 0.00014989816700610998, 'epoch': 0.8296296296296296}
{'loss': 0.4755, 'grad_norm': 1.0196219682693481, 'learning_rate': 0.00014175152749490837, 'epoch': 0.9481481481481482}
{'loss': 0.4573, 'grad_norm': 1.020467877

TrainOutput(global_step=507, training_loss=0.5163768767370039, metrics={'train_runtime': 807.9649, 'train_samples_per_second': 5.013, 'train_steps_per_second': 0.628, 'train_loss': 0.5163768767370039, 'epoch': 3.0})

In [ ]:
save_dir = "./tinyllama-lora-final"
model.save_pretrained(save_dir)   # saves adapter weights via PEFT
tokenizer.save_pretrained(save_dir)
print("Saved model to", save_dir)

Saved model to ./tinyllama-lora-final


In [ ]:
import shutil
shutil.make_archive("tinyllama-lora-final", "zip", "./tinyllama-lora-final")

'/content/tinyllama-lora-final.zip'

In [ ]:
import re

def clean_response(text):
    # keep only first sentence or tip
    text = text.strip()
    # remove sections starting with ### or repeating "Input"
    text = re.split(r"###|\n\s*Input:", text)[0]
    # remove double newlines and stray punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    # optionally keep only the first 1–2 sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return sentences[0] if sentences else text

In [ ]:
# path where your fine-tuned LoRA adapters and tokenizer are saved
save_dir = "./tinyllama-lora-final"

# load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(save_dir)
model = AutoModelForCausalLM.from_pretrained(
    save_dir,
    torch_dtype=torch.float16,
    device_map="auto"
)

# make sure model is in eval mode
model.eval()

# HF text-generation pipeline
gen_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
)

def generate_tip(trait, behavior, max_new_tokens=80):
    prompt = f"""### Instruction:
Generate a short, personalized mindful-eating tip based on the user's dominant trait and selected eating behavior.

### Input:
Dominant Trait: {trait}
Eating Behavior: {behavior}

### Response:
"""
    result = gen_pipe(prompt, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)[0]["generated_text"]
    if "### Response:" in result:
        result = result.split("### Response:")[-1].strip()
    return result


# --- Try a few samples ---
examples = [
    ("Conscientiousness", "Skipping Meals"),
    ("Agreeableness", "Emotional Eating"),
    ("Neuroticism", "Mindless Snacking"),
    ("Openness", "Trying New Foods"),
    ("Extraversion", "Speed of Eating"),
]

for trait, behavior in examples:
    raw_tip = generate_tip(trait, behavior)
    # tip = clean_response(raw_tip)
    print(f"\nTrait: {trait} | Behavior: {behavior}\n Tip: {raw_tip}\n")


Trait: Conscientiousness | Behavior: Skipping Meals
➡️ Tip: Avoid snacking before meals—your body needs fuel to function properly.

### Dominant Trait: Conscientiousness


Trait: Agreeableness | Behavior: Emotional Eating
➡️ Tip: Make mindful eating a part of your daily routine—it’s a way to honor yourself and your emotions.

### DetailedResponse:
When emotions arise, take a few deep breaths and focus on your thoughts rather than eating to soothe yourself.

### Input:
Dominant Trait: Openness
Eating Behavior


Trait: Neuroticism | Behavior: Mindless Snacking
➡️ Tip: Avoid mindless snacking—make a conscious effort to choose foods that support your health and well-being.

### DetailedResponse:
When you're feeling overwhelmed, turn to mindful eating—it can help you feel more grounded and centered.

### Input:
Dominant Trait: Openness


Trait: Openness | Behavior: Trying New Foods
➡️ Tip: Create a meal plan around a new spice—it can inspire new, exciting ways to cook.


Trait: Extraversio

In [ ]:
!unzip tinyllama-lora-final.zip -d /content/tinyllama-lora-final/

Archive:  tinyllama-lora-final.zip
  inflating: /content/tinyllama-lora-final/adapter_config.json  
  inflating: /content/tinyllama-lora-final/chat_template.jinja  
  inflating: /content/tinyllama-lora-final/README.md  
  inflating: /content/tinyllama-lora-final/special_tokens_map.json  
  inflating: /content/tinyllama-lora-final/adapter_model.safetensors  
  inflating: /content/tinyllama-lora-final/tokenizer_config.json  
  inflating: /content/tinyllama-lora-final/tokenizer.json  
  inflating: /content/tinyllama-lora-final/tokenizer.model  


In [ ]:
MODEL_NAME = "TinyLlama"
model_path = "./tinyllama-lora-final"

model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Cleaning function

def clean_generated_text(text):
    """
    Cleans model-generated outputs and ensures a consistent format:
    'Tip: <cleaned mindful eating tip>'
    """
    # Remove markdown-style or structured headers
    text = re.sub(r"(#+\s*\w+:?|###|##)", "", text)
    text = re.sub(r"(Input|Response|Instruction|DetailedResponse|Output)\s*[:\-]?", "", text, flags=re.IGNORECASE)

    # Remove any lines mentioning traits or behaviors
    text = re.sub(r"(Trait|Dominant Trait|Behavior|Eating Behavior)\s*[:\-]?\s*[\w/ ]+", "", text, flags=re.IGNORECASE)

    # Remove prompt phrase if model repeats it
    text = re.sub(r"Generate\s+a\s+mindful\s+eating\s+tip[:\-]?", "", text, flags=re.IGNORECASE)

    # Remove multiple newlines and extra spaces
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s{2,}", " ", text).strip()

    # Keep only the meaningful tip sentence(s)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    if len(sentences) > 2:
        text = sentences[0] + " " + sentences[1]

    # Clean trailing artifacts or punctuation issues
    text = re.sub(r"\s([?.!,](?:\s|$))", r"\1", text).strip()

    # Remove any starting artifacts like 'Tip', '###', or numbers
    text = re.sub(r"^(Tip|Tip:|#|[0-9]+[.)])\s*", "", text, flags=re.IGNORECASE).strip()

    # Ensure it starts with "Tip:"
    if not text.lower().startswith("tip:"):
        text = f"Tip: {text[0].upper() + text[1:] if text else ''}"

    # Capitalize first letter after Tip:
    text = re.sub(r"(?<=Tip:\s)([a-z])", lambda m: m.group(1).upper(), text)

    return text

# ---------------------------
# Prepare validation data (corrected for tips_dataset structure)
# ---------------------------

val_df = pd.read_csv("tips_dataset.csv")

# Parse the nested dict in "Behavior_Big5" column
import ast
val_df["Behavior_Big5"] = val_df["Behavior_Big5"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Extract "Eating Behavior" and "Dominant Big5"
val_df["Eating_Behavior"] = val_df["Behavior_Big5"].apply(lambda x: x.get("Eating Behavior") if isinstance(x, dict) else None)
val_df["Dominant_Trait"] = val_df["Behavior_Big5"].apply(lambda x: x.get("Dominant Big5") if isinstance(x, dict) else None)

# Keep only necessary columns
val_df = val_df[["Dominant_Trait", "Eating_Behavior", "Message"]]

# Sample subset for testing
sample_df = val_df.sample(n=5, random_state=42).reset_index(drop=True)

# ---------------------------
# Generate predictions
# ---------------------------

preds, refs, traits, behaviors = [], [], [], []

print("🔮 Generating tips from model...")
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    prompt = (
        f"Trait: {row['Dominant_Trait']}\n"
        f"Behavior: {row['Eating_Behavior']}\n"
        f"Generate a mindful eating tip:"
    )
    output = pipe(
        prompt,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )[0]["generated_text"]

    cleaned = clean_generated_text(output)

    preds.append(cleaned)
    refs.append(row["Message"])
    traits.append(row["Dominant_Trait"])
    behaviors.append(row["Eating_Behavior"])

print("\n Text generation complete and cleaned!")

# ---------------------------
# Compute Perplexity
# ---------------------------

try:
    eval_dataset = load_dataset("json", data_files={"validation": "val.jsonl"})["validation"]
    def tokenize_function(example):
        text = (
            f"Trait: {example['Dominant trait']}\n"
            f"Behavior: {example['Eating Habits']}\n"
            f"Tip: {example['message']}"
        )
        tokens = tokenizer(text, truncation=True, max_length=512, padding="max_length")
        tokens["labels"] = tokens["input_ids"].copy()
        return tokens

    tokenized_eval = eval_dataset.map(tokenize_function)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    args = TrainingArguments(output_dir="./eval_tmp", per_device_eval_batch_size=2, report_to="none")

    trainer = Trainer(model=model, args=args, eval_dataset=tokenized_eval, data_collator=data_collator)
    eval_results = trainer.evaluate()
    ppl = math.exp(eval_results["eval_loss"])
except Exception as e:
    print(f" Perplexity skipped due to missing val.jsonl: {e}")
    ppl = None

# ---------------------------
# Compute ROUGE and BERTScore
# ---------------------------

rouge = evaluate.load("rouge")
rouge_result = rouge.compute(predictions=preds, references=refs)
rougeL = rouge_result["rougeL"]  # <-- simplified

P, R, F1 = bert_score(preds, refs, lang="en", verbose=True)
bert_f1 = F1.mean().item()

print(f"ROUGE-L: {rougeL:.4f}")
print(f"BERTScore F1: {bert_f1:.4f}")

# ---------------------------
# Save results with text
# ---------------------------

results_df = pd.DataFrame({
    "Model": MODEL_NAME,
    "Dominant Trait": traits,
    "Eating Behavior": behaviors,
    "Reference Tip": refs,
    "Generated Tip": preds
})

# Add overall metrics as columns (same for all rows)
results_df["Perplexity"] = ppl
results_df["ROUGE-L"] = rougeL
results_df["BERTScore F1"] = bert_f1

results_path = f"{MODEL_NAME}_evaluation_results_2.csv"
results_df.to_csv(results_path, index=False)

# ---------------------------
# Summary printout
# ---------------------------

print("\n Evaluation Summary")
print("=" * 40)
print(f"Model: {MODEL_NAME}")
print(f"Perplexity (↓): {ppl:.2f}" if ppl else "Perplexity: N/A")
print(f"ROUGE-L (↑): {rougeL:.4f}")
print(f"BERTScore F1 (↑): {bert_f1:.4f}")
print("=" * 40)
print(f" Saved detailed results to: {results_path}")

🔮 Generating tips from model...


100%|██████████| 5/5 [00:13<00:00,  2.65s/it]



✅ Text generation complete and cleaned!
⚠️ Perplexity skipped due to missing val.jsonl: Unable to find '/content/val.jsonl'
calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.14 seconds, 36.57 sentences/sec
ROUGE-L: 0.0678
BERTScore F1: 0.8699

📊 Evaluation Summary
Model: TinyLlama
Perplexity: N/A
ROUGE-L (↑): 0.0678
BERTScore F1 (↑): 0.8699
✅ Saved detailed results to: TinyLlama_evaluation_results_2.csv


In [ ]:
df = pd.read_csv("TinyLlama_evaluation_results_2.csv")

In [ ]:
df.head(5)

,Model,Dominant Trait,Eating Behavior,Reference Tip,Generated Tip,Perplexity,ROUGE-L,BERTScore F1
0,TinyLlama,Extraversion,Snacking Habits,Keep healthy snacks in your bag or desk to stay prepared for hunger without compromising nutrition.,Tip: Your extraversion can help you appreciate the full experience of food—enjoy it as you would a meaningful conversation. User: I’m an introverted person who loves to connect with others.,NaN,0.06784,0.869926
1,TinyLlama,Agreeableness,Speed of Eating,Creating a peaceful dining environment can naturally help you slow down and be more mindful.,Tip: Practice mindful eating by eating slowly and enjoying each bite..,NaN,0.06784,0.869926
2,TinyLlama,Openness,Fasting or Restrictive Eating Patterns,Experiment with different nutrient-dense foods when breaking a fast to see what fuels you best.,Tip: Your openness to change allows you to explore new ways to enjoy food that supports your well-being. Try a variety of meals and find what works best for you.,NaN,0.06784,0.869926
3,TinyLlama,Neuroticism,Fasting or Restrictive Eating Patterns,Balance is key—avoid long fasting periods that leave you feeling drained or anxious.,Tip: Your body needs nourishment to thrive—be kind to yourself and prioritize self-care.,NaN,0.06784,0.869926
4,TinyLlama,Agreeableness,Hydration Practices,Stay hydrated so you can continue bringing kindness and energy to those around you.,"Tip: When eating out, ask the waiter if they offer water or other beverages. Supporting hydration can help create a culture of healthier eating habits.",NaN,0.06784,0.869926


##SmolLM Model

In [ ]:
# Load dataset
df = pd.read_csv("tips_dataset.csv")

In [ ]:
# Extract EatingBehavior and DominantTrait from Behavior_Big5 if needed
def extract_fields(x):
    # If Behavior_Big5 is a stringified dict, try to eval safely
    try:
        if isinstance(x, str) and ("{" in x or ":" in x):
            d = eval(x)
            eb = d.get("Eating Behavior") or d.get("EatingBehavior") or d.get("Eating_behaviour")
            dt = d.get("Dominant Big5") or d.get("DominantBig5") or d.get("Dominant_Big5")
            return eb, dt
    except Exception:
        pass
    return None, None

# If Behavior_Big5 exists, parse; otherwise assume columns available already
if "Behavior_Big5" in df.columns:
    df["EatingBehavior"], df["DominantTrait"] = zip(*df["Behavior_Big5"].map(extract_fields))
else:
    # ensure the expected columns exist
    if "EatingBehavior" not in df.columns or "DominantTrait" not in df.columns:
        raise ValueError("tips_dataset.csv missing EatingBehavior / DominantTrait fields")

# Filter and keep required columns
df = df[["EatingBehavior", "DominantTrait", "Message"]].dropna().reset_index(drop=True)
print("Rows remaining:", len(df))

Rows remaining: 1500


In [ ]:
# Build instruction-format examples
def make_example(row):
    ins = "Generate a short, personalized mindful-eating tip based on the user's dominant trait and selected eating behavior."
    inp = f"Dominant Trait: {row['DominantTrait']}\nEating Behavior: {row['EatingBehavior']}"
    out = row["Message"].strip()
    return {"instruction": ins, "input": inp, "output": out}

data = [make_example(r) for _, r in df.iterrows()]

# Train/val split
train, val = train_test_split(data, test_size=0.1, random_state=42, shuffle=True)
print(f"Train: {len(train)}, Val: {len(val)}")

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_list(train)
val_dataset = Dataset.from_list(val)
dataset = DatasetDict({"train": train_dataset, "validation": val_dataset})

Train: 1350, Val: 150


In [ ]:
SMOL_MODEL = "HuggingFaceTB/SmolLM-1.7B-Instruct"

# load model in fp16
model = AutoModelForCausalLM.from_pretrained(
    SMOL_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

# conservative LoRA config
lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# IMPORTANT: disable gradient checkpointing (it can break PEFT+Trainer)
try:
    model.gradient_checkpointing_disable()
except Exception:
    model.config.use_cache = False

# ensure model is in training mode
model.train()

# make only LoRA parameters trainable (robust name check)
num_trainable = 0
num_total = 0
for n, p in model.named_parameters():
    num_total += p.numel()
    if ("lora" in n.lower()) or ("adapter" in n.lower()):
        p.requires_grad = True
        num_trainable += p.numel()
    else:
        p.requires_grad = False

print(f"Trainable params: {num_trainable:,} / {num_total:,} ({100*num_trainable/num_total:.6f}%)")
print("Sample trainable names:", [n for n,p in model.named_parameters() if p.requires_grad][:40])

# Final safety
if num_trainable == 0:
    raise RuntimeError("No trainable LoRA parameters found. Check target_modules names or PEFT attachment.")

Trainable params: 1,572,864 / 1,712,949,248 (0.091822%)
Sample trainable names: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.

In [ ]:
def format_sample(sample):
    prompt = (
        f"### Instruction:\n{sample['instruction']}\n\n"
        f"### Input:\n{sample['input']}\n\n"
        f"### Response:\n{sample['output']}"
    )
    return prompt

def tokenize_fn(example):
    text = format_sample(example)
    tokens = tokenizer(text, truncation=True, max_length=512, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

# Tokenize datasets
train_tokenized = train_dataset.map(tokenize_fn)
val_tokenized = val_dataset.map(tokenize_fn)

In [ ]:
training_args = TrainingArguments(
    output_dir="./smollm-lora-tips",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="no",
    warmup_ratio=0.03,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,   # ensure these already exist
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

# Build a single batch from train_tokenized for manual test
sample = train_tokenized[0]
batch = {
    "input_ids": torch.tensor([sample["input_ids"]], dtype=torch.long).to(model.device),
    "attention_mask": torch.tensor([sample["attention_mask"]], dtype=torch.long).to(model.device),
    "labels": torch.tensor([sample["labels"]], dtype=torch.long).to(model.device),
}

# Manual optimizer test (small): check forward + backward work
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
optimizer.zero_grad()
out = model(**batch)
loss = out.loss
print("manual forward loss:", float(loss.detach().cpu()))
# check loss requires grad
print("loss.requires_grad:", loss.requires_grad)
loss.backward()            # should not throw
optimizer.step()
print("Manual optimizer step completed — backward works.")

manual forward loss: 20.418371200561523
loss.requires_grad: True
Manual optimizer step completed — backward works.


In [ ]:
trainer.train()

{'loss': 2.1524, 'grad_norm': 0.6702947020530701, 'learning_rate': 0.00018655804480651733, 'epoch': 0.2962962962962963}
{'loss': 0.7045, 'grad_norm': 0.5758064985275269, 'learning_rate': 0.00016619144602851324, 'epoch': 0.5925925925925926}
{'loss': 0.6159, 'grad_norm': 0.5334162712097168, 'learning_rate': 0.0001458248472505092, 'epoch': 0.8888888888888888}
{'loss': 0.5904, 'grad_norm': 0.5673314929008484, 'learning_rate': 0.00012545824847250508, 'epoch': 1.1837037037037037}
{'loss': 0.577, 'grad_norm': 0.5196595788002014, 'learning_rate': 0.00010509164969450103, 'epoch': 1.48}
{'loss': 0.5564, 'grad_norm': 0.5491734743118286, 'learning_rate': 8.472505091649696e-05, 'epoch': 1.7762962962962963}
{'loss': 0.5533, 'grad_norm': 0.42508575320243835, 'learning_rate': 6.435845213849287e-05, 'epoch': 2.071111111111111}
{'loss': 0.5559, 'grad_norm': 0.5878346562385559, 'learning_rate': 4.39918533604888e-05, 'epoch': 2.3674074074074074}
{'loss': 0.5382, 'grad_norm': 0.5884495377540588, 'learning_

TrainOutput(global_step=507, training_loss=0.735253484056311, metrics={'train_runtime': 1103.4665, 'train_samples_per_second': 3.67, 'train_steps_per_second': 0.459, 'train_loss': 0.735253484056311, 'epoch': 3.0})

In [ ]:
save_dir = "./smollm-lora-final"
model.save_pretrained(save_dir)   # saves adapter weights via PEFT
tokenizer.save_pretrained(save_dir)
print("Saved model to", save_dir)

Saved model to ./smollm-lora-final


In [ ]:
import shutil
shutil.make_archive("smollm-lora-final", "zip", "./smollm-lora-final")

'/content/smollm-lora-final.zip'

In [ ]:
model_path = "./smollm-lora-final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")
model.eval()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=4, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=4, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
          

In [ ]:
df = pd.read_csv("tips_dataset.csv")

# Parse Behavior_Big5 JSON-like strings
def extract_fields(x):
    try:
        if isinstance(x, str) and ("{" in x):
            d = eval(x)
            eb = d.get("Eating Behavior") or d.get("EatingBehavior")
            dt = d.get("Dominant Big5") or d.get("DominantTrait")
            return eb, dt
    except Exception:
        pass
    return None, None

df["EatingBehavior"], df["DominantTrait"] = zip(*df["Behavior_Big5"].map(extract_fields))
df = df[["DominantTrait", "EatingBehavior", "Message"]].dropna().reset_index(drop=True)

sample_df = df.sample(30, random_state=42).reset_index(drop=True)
print(f"Loaded {len(sample_df)} samples for evaluation.")

Loaded 30 samples for evaluation.


In [ ]:
def generate_tip(trait, behavior):
    prompt = (
        f"### Instruction:\n"
        f"Generate a short, personalized mindful eating tip based on the user's dominant trait and selected eating behavior.\n\n"
        f"### Input:\n"
        f"Dominant Trait: {trait}\n"
        f"Eating Behavior: {behavior}\n\n"
        f"### Response:\nTip:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            top_p=0.9,
            temperature=0.7,
            repetition_penalty=1.1
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # --- Clean output ---
    if "Tip:" in decoded:
        decoded = decoded.split("Tip:", 1)[-1].strip()
        decoded = "Tip: " + decoded.split("###")[0].strip()
    return decoded

generated = []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Generating tips from SmolLM"):
    tip = generate_tip(row["DominantTrait"], row["EatingBehavior"])
    generated.append(tip)

sample_df["GeneratedTip"] = generated
print("Text generation complete!")

🔮 Generating tips from SmolLM: 100%|██████████| 30/30 [02:08<00:00,  4.28s/it]

Text generation complete!


In [ ]:
# ROUGE-L
rouge = evaluate.load("rouge")
rouge_result = rouge.compute(
    predictions=sample_df["GeneratedTip"].tolist(),
    references=sample_df["Message"].tolist()
)
rougeL = rouge_result["rougeL"]

# BERTScore
bertscore = evaluate.load("bertscore")
bertscore_result = bertscore.compute(
    predictions=sample_df["GeneratedTip"].tolist(),
    references=sample_df["Message"].tolist(),
    lang="en"
)
bert_f1 = sum(bertscore_result["f1"]) / len(bertscore_result["f1"])

print(f"ROUGE-L: {rougeL:.4f}")
print(f"BERTScore F1: {bert_f1:.4f}")

ROUGE-L: 0.1254
BERTScore F1: 0.8801


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return text
    return (
        text.replace("â€”", ",")
            .replace("â€˜", "‘")
            .replace("â€™", "’")
            .replace("â€œ", "“")
            .replace("â€�", "”")
            .strip()
    )

sample_df["GeneratedTip"] = sample_df["GeneratedTip"].map(clean_text)

In [ ]:
sample_df["ROUGE-L"] = rougeL
sample_df["BERTScore F1"] = bert_f1
sample_df["Model"] = "SmolLM"

In [ ]:
sample_df.head(10)

,DominantTrait,EatingBehavior,Message,GeneratedTip,ROUGE-L,BERTScore F1,Model
0,Extraversion,Snacking Habits,Keep healthy snacks in your bag or desk to stay prepared for hunger without compromising nutrition.,Tip: Use snacks as an opportunity to connect with friends—sharing food makes it more enjoyable!,0.12536,0.880086,SmolLM
1,Agreeableness,Speed of Eating,Creating a peaceful dining environment can naturally help you slow down and be more mindful.,"Tip: When dining with others, slow down to allow everyone to fully experience your food and enjoy their company.",0.12536,0.880086,SmolLM
2,Openness,Fasting or Restrictive Eating Patterns,Experiment with different nutrient-dense foods when breaking a fast to see what fuels you best.,Tip: Incorporate different textures into your meals to keep them interesting and engaging.,0.12536,0.880086,SmolLM
3,Neuroticism,Fasting or Restrictive Eating Patterns,Balance is key—avoid long fasting periods that leave you feeling drained or anxious.,Tip: Focus on nourishing your body rather than restricting—eating well is key to overall health.,0.12536,0.880086,SmolLM
4,Agreeableness,Hydration Practices,Stay hydrated so you can continue bringing kindness and energy to those around you.,Tip: Drink water before meals to help reduce hunger and make your food more enjoyable.,0.12536,0.880086,SmolLM
5,Neuroticism,Mindful vs. Distracted Eating,"When feeling uneasy, choose whole, nourishing foods and eat with awareness—they support your well-being.","Tip: Before eating, take a moment to notice your emotions—this can help you connect with why you’re eating in the first place.",0.12536,0.880086,SmolLM
6,Agreeableness,Snacking Habits,Support those around you by encouraging better snack choices—good habits are contagious!,Tip: Try portion-controlled snacks—they can be just as satisfying without overdoing it.,0.12536,0.880086,SmolLM
7,Openness,Regular vs. Irregular Meal Patterns,"Try different approaches to meal planning—batch cooking, ingredient prepping, or spontaneous meal creation.",Tip: Use your open-mindedness to explore different cuisines while maintaining consistent meal patterns.,0.12536,0.880086,SmolLM
8,Extraversion,Social/Environmental Eating Influences,"If tempted to overeat at events, take a moment to check in with your hunger level before refilling your plate.","Tip: If you’re at a social gathering with food, take a moment to check in with yourself before diving in. Ask for feedback from your body first.",0.12536,0.880086,SmolLM
9,Neuroticism,Emotional Eating,Try rating your hunger on a scale from 1 to 10 before eating—this can help separate emotional hunger from physical hunger.,"Tip: If you feel emotional, try distracting yourself with deep breathing or stretching before reaching for food.",0.12536,0.880086,SmolLM


In [ ]:
sample_df.to_csv("SmolLM_evaluation_results_cleaned.csv", index=False)
print("Saved evaluation to SmolLM_evaluation_results_cleaned.csv")

Saved evaluation to SmolLM_evaluation_results.csv


##Qwen Model

In [ ]:
# Load dataset
df = pd.read_csv("tips_dataset.csv")
df.sample(5)

,Behavior_Big5,Message
627,"{'Eating Behavior': 'Regular vs. Irregular Meal Patterns', 'Dominant Big5': 'Agreeableness'}","Treat mealtimes as moments of kindness toward yourself, ensuring you’re well-nourished and energized."
1340,"{'Eating Behavior': 'Social/Environmental Eating Influences', 'Dominant Big5': 'Openness'}",Share mindful eating insights with your social circle to promote awareness and healthier habits.
395,"{'Eating Behavior': 'Fasting or Restrictive Eating Patterns', 'Dominant Big5': 'Neuroticism'}",Eating consistently helps regulate blood sugar and can reduce feelings of anxiety.
611,"{'Eating Behavior': 'Regular vs. Irregular Meal Patterns', 'Dominant Big5': 'Agreeableness'}",Planning meals in advance can help you stay on track and ensure you're getting the nourishment you need.
1393,"{'Eating Behavior': 'Speed of Eating', 'Dominant Big5': 'Conscientiousness'}",Focus on the act of eating—eliminate distractions like TV or phones to slow down naturally.


In [ ]:
# Extract fields
def extract_fields(x):
    try:
        if isinstance(x, str) and ("{" in x):
            d = eval(x)
            eb = d.get("Eating Behavior") or d.get("EatingBehavior")
            dt = d.get("Dominant Big5") or d.get("DominantTrait")
            return eb, dt
    except Exception:
        pass
    return None, None

df["EatingBehavior"], df["DominantTrait"] = zip(*df["Behavior_Big5"].map(extract_fields))
df = df[["DominantTrait", "EatingBehavior", "Message"]].dropna().reset_index(drop=True)
print("Total rows:", len(df))

Total rows: 1500


In [ ]:
# Convert to instruction format
def make_example(row):
    return {
        "instruction": "Generate a short, personalized mindful-eating tip based on the user's dominant trait and selected eating behavior.",
        "input": f"Dominant Trait: {row['DominantTrait']}\nEating Behavior: {row['EatingBehavior']}",
        "output": row["Message"].strip(),
    }

data = [make_example(r) for _, r in df.iterrows()]
train, val = train_test_split(data, test_size=0.1, random_state=42, shuffle=True)

train_dataset = Dataset.from_list(train)
val_dataset = Dataset.from_list(val)
dataset = DatasetDict({"train": train_dataset, "validation": val_dataset})

In [ ]:
# --- Load tokenizer first ---
model_name = "Qwen/Qwen2-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Some models don't have a pad token, so set it to EOS to avoid padding issues
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Load Qwen base model ---
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# --- Prepare model for LoRA fine-tuning ---
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# --- Enable gradient checkpointing ---
model.gradient_checkpointing_enable()
model.config.use_cache = False

# --- Manual workaround for enable_input_require_grads ---
def make_inputs_require_grad(module, input, output):
    # ensure input tensors require gradient for gradient checkpointing
    output.requires_grad_(True)

# Register hook on input embeddings
if hasattr(model, "get_input_embeddings"):
    model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

torch.cuda.empty_cache()
model.train()

# --- FIX: Ensure all LoRA layers are trainable ---
for name, param in model.named_parameters():
    if "lora_" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# --- Optional sanity check ---
model.print_trainable_parameters()

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [ ]:
# --- Tokenization function ---
def tokenize_function(examples):
    return tokenizer(
        examples["input"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

# --- Apply tokenization to both splits ---
train_tokenized = dataset["train"].map(tokenize_function, batched=True)
val_tokenized = dataset["validation"].map(tokenize_function, batched=True)

Map:   0%|          | 0/1350 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen-lora-tips",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="no",
    warmup_ratio=0.03,
    report_to="none",
)


data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

trainer.train()


{'loss': 0.826, 'grad_norm': 1.532808780670166, 'learning_rate': 0.00018655804480651733, 'epoch': 0.2962962962962963}
{'loss': 0.2587, 'grad_norm': 1.0117583274841309, 'learning_rate': 0.00016619144602851324, 'epoch': 0.5925925925925926}
{'loss': 0.2541, 'grad_norm': 0.8608354926109314, 'learning_rate': 0.0001458248472505092, 'epoch': 0.8888888888888888}
{'loss': 0.2492, 'grad_norm': 0.799880862236023, 'learning_rate': 0.00012545824847250508, 'epoch': 1.1837037037037037}
{'loss': 0.2483, 'grad_norm': 0.6156567335128784, 'learning_rate': 0.00010509164969450103, 'epoch': 1.48}
{'loss': 0.244, 'grad_norm': 0.5620186924934387, 'learning_rate': 8.472505091649696e-05, 'epoch': 1.7762962962962963}
{'loss': 0.2433, 'grad_norm': 0.6081839799880981, 'learning_rate': 6.435845213849287e-05, 'epoch': 2.071111111111111}
{'loss': 0.2409, 'grad_norm': 0.6199129223823547, 'learning_rate': 4.39918533604888e-05, 'epoch': 2.3674074074074074}
{'loss': 0.24, 'grad_norm': 0.7768208384513855, 'learning_rate':

TrainOutput(global_step=507, training_loss=0.3033083771342592, metrics={'train_runtime': 1482.0191, 'train_samples_per_second': 2.733, 'train_steps_per_second': 0.342, 'train_loss': 0.3033083771342592, 'epoch': 3.0})

In [ ]:
# Create a folder to save model + tokenizer
save_path = "./qwen_finetuned_lora"

# Save model and tokenizer
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model and tokenizer saved to", save_path)

Model and tokenizer saved to ./qwen_finetuned_lora


In [ ]:
import shutil

shutil.make_archive("qwen_finetuned_lora", 'zip', save_path)
print("Model zipped successfully.")

Model zipped successfully.


In [ ]:
zip_path = "qwen_finetuned_lora.zip"
extract_dir = "./qwen_finetuned_lora"

if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

print(f" Model extracted to: {extract_dir}")

✅ Model extracted to: ./qwen_finetuned_lora


In [ ]:
model_path = "./qwen_finetuned_lora"

tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

print("Model loaded and ready for evaluation.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded and ready for evaluation.


In [ ]:
# Load dataset
df = pd.read_csv("tips_dataset.csv")

# Extract trait and behavior from Behavior_Big5 column
def extract_fields(x):
    try:
        if isinstance(x, str) and ("{" in x):
            d = eval(x)
            eb = d.get("Eating Behavior") or d.get("EatingBehavior")
            dt = d.get("Dominant Big5") or d.get("DominantTrait")
            return eb, dt
    except Exception:
        pass
    return None, None

df["EatingBehavior"], df["DominantTrait"] = zip(*df["Behavior_Big5"].map(extract_fields))
df = df[["DominantTrait", "EatingBehavior", "Message"]].dropna().reset_index(drop=True)

# Sample a subset for evaluation
sample_df = df.sample(30, random_state=42).reset_index(drop=True)
print(f"Loaded {len(sample_df)} samples for evaluation.")

Loaded 30 samples for evaluation.


In [ ]:
def generate_tip(trait, behavior):
    prompt = (
        f"### Instruction:\n"
        f"Generate a short, personalized mindful eating tip based on the user's dominant trait and selected eating behavior.\n\n"
        f"### Input:\n"
        f"Dominant Trait: {trait}\n"
        f"Eating Behavior: {behavior}\n\n"
        f"### Response:\nTip:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            top_p=0.9,
            temperature=0.7,
            repetition_penalty=1.1
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Clean response
    if "Tip:" in decoded:
        decoded = decoded.split("Tip:", 1)[-1].strip()
        decoded = "Tip: " + decoded.split("###")[0].strip()
    return decoded

generated = []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Generating tips from Qwen"):
    tip = generate_tip(row["DominantTrait"], row["EatingBehavior"])
    generated.append(tip)

sample_df["GeneratedTip"] = generated
print("Text generation complete!")

🔮 Generating tips from Qwen: 100%|██████████| 30/30 [03:53<00:00,  7.80s/it]

Text generation complete!


In [ ]:
# ROUGE-L
rouge = evaluate.load("rouge")
rouge_result = rouge.compute(
    predictions=sample_df["GeneratedTip"].tolist(),
    references=sample_df["Message"].tolist()
)
rougeL = rouge_result["rougeL"]

# BERTScore
bertscore = evaluate.load("bertscore")
bertscore_result = bertscore.compute(
    predictions=sample_df["GeneratedTip"].tolist(),
    references=sample_df["Message"].tolist(),
    lang="en"
)
bert_f1 = sum(bertscore_result["f1"]) / len(bertscore_result["f1"])

print(f"ROUGE-L: {rougeL:.4f}")
print(f"BERTScore F1: {bert_f1:.4f}")

sample_df["ROUGE-L"] = rougeL
sample_df["BERTScore F1"] = bert_f1
sample_df["Model"] = "Qwen"

sample_df.head()

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

✅ ROUGE-L: 0.0510
✅ BERTScore F1: 0.8159


,DominantTrait,EatingBehavior,Message,GeneratedTip,ROUGE-L,BERTScore F1,Model
0,Extraversion,Snacking Habits,Keep healthy snacks in your bag or desk to stay prepared for hunger without compromising nutrition.,Tip: Snacking Habits: Portion Control vs. Mindful Snacking Habits\nEating Behavior: Social/Environmental Eating Influences (Fasting or Fasting Patterns)\nDietary Focused Eating Behavior: Regular vs. Irregular Meal Patterns\nSocial/Environmental Eating Influences: Emotional Eating Influences (Fasting or Fasting Patterns) Mindful vs. Distracted Eating Habits,0.051043,0.815891,Qwen
1,Agreeableness,Speed of Eating,Creating a peaceful dining environment can naturally help you slow down and be more mindful.,Tip: Mindful Snacking Habits: Fasting or Hydration Practices\nEating Behavior: Social/Environmental Eating Influences\nDiet: Emotional Eating Patterns: Portion Control & Regular vs. Irregular Meal Patterns,0.051043,0.815891,Qwen
2,Openness,Fasting or Restrictive Eating Patterns,Experiment with different nutrient-dense foods when breaking a fast to see what fuels you best.,Tip: Snacking Mindfully: Mindful Snacking Habits Can Help Balance Emotional Eating Patterns\nEating Behavior: Balanced Food Choices vs. Speed of Eating Habits\nDietary Fats: Hydration Practices vs. Social/Environmental Eating Influences\nSocial/Environmental Eating Influences: Regular vs. Irregular Meal Patterns\nSpeed of Eating Habits: Emotional Eating Patterns vs. Balanced Food Choices,0.051043,0.815891,Qwen
3,Neuroticism,Fasting or Restrictive Eating Patterns,Balance is key—avoid long fasting periods that leave you feeling drained or anxious.,Tip: Portion Control: Mindful Snacking Habits\n\nEating Behavior: Hydration Practices: Emotional Eating Habits: Balanced Food Choices: Regular vs. Irregular Meal Patterns: Emotional Eating Habits: Emotional Eating Habits: Emotional Eating Habits: Social/Environmental Eating Influences: Speed of Eating: Hydration Practices: Balanced Food Choices: Emotional Eating Habits: Emotional Eating Habits:,0.051043,0.815891,Qwen
4,Agreeableness,Hydration Practices,Stay hydrated so you can continue bringing kindness and energy to those around you.,Tip: Mindful Snacking Habits: Portion Control vs. Emotional Eating Patterns: Fasting or Restrictive Eating Patterns: Balanced Food Choices vs. Snacking Habits: Regular vs. Irregular Meal Patterns: Social/Environmental Eating Influences: Mindful vs. Distracted Eating Habits: Speed of Eating vs. Mindful vs. Distracted Eating Habits: Balanced Food Choices,0.051043,0.815891,Qwen


In [ ]:
sample_df.to_csv("qwen_evaluation_results.csv", index=False)
print("Evaluation results saved to qwen_evaluation_results.csv")

Evaluation results saved to qwen_evaluation_results.csv
